<a href="https://colab.research.google.com/github/jaydenchoe/python-lecture-jumptophython-examples/blob/main/2025-1115_gemini_api_file_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠<br>Gemini, 내 문서를 읽어줘!<br>File Search (RAG) API 왕초보 가이드

안녕하세요! 챗봇 '제미'에게 '기억력'을 심어주는 두 번째 시간입니다.

지난 시간에 배운 '채팅 기록(History)'은 '제미'가 *방금 나눈 대화*를 기억하는 기술이었죠?

이번에 배울 **'파일 검색(File Search)' API**는, '제미'에게 아예 **'참고 자료(파일)'**를 통째로 읽혀서, 그 파일의 내용을 기반으로 대답하게 만드는 궁극의 '기억력' 기술입니다! (이것을 전문 용어로 **RAG**라고 불러요.)

이 기술을 사용하면 '제미'가...
* 여러분이 업로드한 PDF, TXT, DOCX, HWP 파일을 읽고 요약하게 할 수 있어요.
* 여러분의 '회사 내부 문서'나 '수업 자료'를 학습해서, 똑똑한 Q&A 봇이 될 수 있어요.

자, '제미'에게 여러분의 파일을 읽히는 방법을 배워봅시다! (2025년 11월 최신)

## 1단계: 챗봇을 위한 '두뇌' 설치 및 '열쇠' 등록하기 🧠🔑

모든 것의 시작이죠! '제미'의 두뇌(`google-generativeai`)를 설치하고, 코랩의 '비밀' 기능(🔑)에 `GOOGLE_API_KEY`를 등록해 주세요.

In [ ]:
!pip uninstall -y google-generativeai
!pip install -U google-genai

In [ ]:
from google import genai
from google.genai import types
import time
import os

# 🤫 코랩의 '비밀' 기능(userdata)에서 API 키를 가져옵니다.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✨ '제미'가 깨어날 준비가 되었어요! (비밀 키 사용 완료)")
except Exception as e:
    print(f"🔑 비밀 키를 가져오는 데 실패했어요. {e}")
    print("--- (경고) --- \n왼쪽 🔑 아이콘에서 'GOOGLE_API_KEY'라는 이름으로 비밀 키를 꼭 저장해주세요!")

## 2단계: 파일 '기억 저장소' 만들기 (FileSearchStore)

'제미'가 파일을 기억하게 하려면, 파일을 보관할 **'기억 저장소(FileSearchStore)'**가 필요해요.

이 저장소는 파일의 텍스트를 그냥 보관하는 게 아니라, '제미'가 빠르게 검색할 수 있도록 '핵심 의미(임베딩)'를 추출해서 영구적으로 보관하는 특별한 공간입니다.

**[중요]** '파일 검색' 기능을 사용하려면 `genai.Client()`라는 특별한 '관리자' 객체가 필요합니다!

In [ ]:

client = genai.Client( api_key= API_KEY)

# 파일 검색 저장소 생성
file_search_store = client.file_search_stores.create(
    config={'display_name': '나의 첫 문서 저장소'}
)

print(f"{file_search_store.display_name} 저장소가 생성되었습니다.")
print(f"(고유 ID 이름: {file_search_store.name})")


## 3단계: '기억 저장소'에 파일 업로드하기 (upload_to_file_search_store)

이제 '제미'에게 읽힐 파일을 만들고, 방금 만든 '기억 저장소'에 업로드해 볼게요.

`upload_to_file_search_store` 함수를 쓰면 **(1) 파일 업로드 + (2) 파일 분석(청크) + (3) 의미 추출(색인)**을 한 번에 처리해 줍니다.

파일을 분석하는 데 시간이 좀 걸리기 때문에, `operation.done`을 체크하며 '제미'가 파일을 다 읽을 때까지 기다려야 해요.

In [ ]:
# 1. 코랩에 'sample_memo.txt'라는 이름의 예제 파일 만들기
# (1934년에 나온 '나는 클라우디우스다' 책에 대한 메모입니다)
with open("sample_memo.txt", "w") as f:
    f.write( "로버트 그레이브스(Robert Graves)는 1934년에 '나는 클라우디우스다'라는 소설을 썼습니다.\n")
    f.write("그는 시인이자 고전학자였습니다.\n")
    f.write("이 메모는 2025년 11월 15일에 작성되었습니다.")
print("'sample_memo.txt' 파일 생성 완료!")

# 2. 파일을 '저장소'에 직접 업로드하고 '색인'까지 요청
print(f"\n'{file_search_store.name}' 저장소에 'sample_memo.txt' 파일을 업로드하고 색인을 시작합니다...")
operation = client.file_search_stores.upload_to_file_search_store(
    file='sample_memo.txt', # 업로드할 파일 경로
    file_search_store_name=file_search_store.name, # 어느 저장소에 넣을지
    config={'display_name': '오늘의 메모'} # 이 파일의 별명
)

# 3. "제미"가 파일을 다 읽고 정리할 때까지 기다리기!
print("\n파일 처리 중... (최대 몇 분 걸릴 수 있습니다)")
while not operation.done:
    print("...아직 처리 중...")
    time.sleep(5) # 5초 대기
    operation = client.operations.get(operation) # 진행 상황 다시 확인

print(f"\n🎉 파일 처리가 완료되었습니다!")

## 4단계: '기억'에 대해 질문하기 (RAG 실행!)

드디어 '제미'가 파일을 다 읽었습니다!

이제 '제미'에게 **`tools`** 라는 '도구'를 쥐여주고, "파일을 검색해서 대답해!"라고 요청해 봅시다. (이것이 RAG의 핵심입니다!)

`gemini-2.5-flash` 모델도 파일 검색을 완벽하게 지원합니다.

In [ ]:


tools = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[file_search_store.name]
        )
    )
]

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="로버트 그레이브스에 대해 알려줘",
    config=types.GenerateContentConfig(tools=tools)
)

print(response.text)


## 5단계: "어떻게 알았어?" (답변 근거 확인하기 - Citations)

'제미'는 놀랍게도 답변의 근거가 된 **'원본 구절(인용)'**을 알려줄 수 있습니다!

답변 객체 `response`에서 `candidates[0].grounding_metadata`를 확인하면, '제미'가 어떤 파일의 어떤 구절을 보고 대답했는지 알 수 있어요.

In [ ]:
if response.candidates[0].grounding_metadata:
    print("--- 📚 '제미'가 답변의 근거를 찾았습니다! ---")
    print(response.candidates[0].grounding_metadata)
else:
    print("--- 📚 답변의 근거를 찾지 못했습니다. ---" )

## 6단계 (고급): 파일에 '태그' 붙여서 관리하기 (Metadata)

'기억 저장소'에 파일이 수백 개가 되면 헷갈리겠죠?

파일을 업로드(또는 '가져오기')할 때 **`custom_metadata`**라는 '태그'를 붙여서 파일을 쉽게 분류하고 검색할 수 있습니다.

(이번에는 파일을 (1)먼저 업로드하고 (2)나중에 '가져오기(import)'하는 두 단계 방식을 써볼게요.)

In [ ]:
# 1. 태그를 붙일 파일 2개 생성 (다른 책 정보)
with open("book_1955.txt", "w") as f:
    f.write("'그리스 신화'는 로버트 그레이브스가 1955년에 출간했습니다.")

with open("book_1985.txt", "w") as f:
    f.write("'파운데이션과 지구'는 아이작 아시모프가 1985년에 출간했습니다.")
print("파일 2개 생성 완료: book_1955.txt, book_1985.txt")

# 2. Files API로 파일 2개를 먼저 업로드 (이건 '임시' 파일입니다)
sample_file_1 = client.files.upload(file='book_1955.txt', config={'name': 'book-1955-7'})
sample_file_2 = client.files.upload(file='book_1985.txt', config={'name': 'book-1985-7'})
print(f"\n임시 파일 2개 업로드 완료: {sample_file_1.name}, {sample_file_2.name}")

# 3. '저장소'로 '가져오기' 하면서 '태그(metadata)'를 붙입니다.
print("파일을 저장소로 가져오면서 '태그'를 붙입니다...")

op1 = client.file_search_stores.import_file(
    file_search_store_name=file_search_store.name,
    file_name=sample_file_1.name,
    config={
        "custom_metadata": [
            {"key": "author", "string_value": "Robert Graves"},
            {"key": "year", "numeric_value": 1955},
        ]
    },
)

op2 = client.file_search_stores.import_file(
    file_search_store_name=file_search_store.name,
    file_name=sample_file_2.name,
    config={
        "custom_metadata": [
            {"key": "author", "string_value": "Isaac Asimov"},
            {"key": "year", "numeric_value": 1985},
        ]
    },
)

# 두 파일이 모두 처리될 때까지 기다립니다.
print("\n태그 붙인 파일 처리 중...")
while not op1.done or not op2.done:
    print("...")
    time.sleep(5)
    if not op1.done: op1 = client.operations.get(op1)
    if not op2.done: op2 = client.operations.get(op2)

print("🎉 태그(Metadata)가 적용된 파일 2개 처리 완료!")

## 7단계 (고급): '태그'로 '필터링'해서 질문하기

이제 '제미'를 조종해서 "'author' 태그가 'Isaac Asimov'인 파일" *에서만* 검색하도록 만들 수 있습니다!

도구 설정에서 **`metadata_filter`**를 사용하면 됩니다.

In [ ]:
# 'tools'에 'metadata_filter'를 추가합니다.
tools_with_filter = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[file_search_store.name],
            # 권장 형식  author="Isaac Asimov"
            metadata_filter='author="Isaac Asimov"',
        )
    )
]

# (모델은 위에서 만든 'gemini-2.5-flash'를 사용합니다)

# 1. '로버트 그레이브스'에 대해 물어보면?
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="로버트 그레이브스의 책이 뭐야?",
    config=types.GenerateContentConfig(tools=tools_with_filter),
)

print("--- (필터: '아시모프') '그레이브스'에 대해 질문한 결과 ---")
print(f"🤖: {response.text}\n(아마 모른다고 대답할 거예요! '아시모프' 파일만 검색했으니까요.)")

# 2. '아이작 아시모프'에 대해 물어보면?
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="아이작 아시모프의 책이 뭐야?",
    config=types.GenerateContentConfig(tools=tools_with_filter)
)
print("\n--- (필터: '아시모프') '아시모프'에 대해 질문한 결과 ---")
print(f"🤖: {response.text}")

## 8단계 (고급): 파일 '조각(Chunk)' 크기 조절하기

'제미'는 사실 파일을 통째로 읽는 게 아니라, 잘게 **'조각(Chunk)'**으로 나눠서 읽어요.

**`chunking_config`**를 설정하면 '조각'의 크기(`max_tokens_per_chunk`)와 '조각'이 겹치는 정도(`max_overlap_tokens`)를 직접 조절할 수 있습니다. (문서가 복잡할 때 유용해요!)

In [ ]:
# 예: 파일을 200토큰 크기로 자르고, 20토큰씩 겹치게 설정
custom_chunk_config = {
    'chunking_config': {
        'white_space_config': {
            'max_tokens_per_chunk': 200,
            'max_overlap_tokens': 20
        }
    }
}

# (실제 업로드 시 이렇게 사용합니다)
# operation = client.file_search_stores.upload_to_file_search_store(
#     file='my_large_file.pdf',
#     file_search_store_name=file_search_store.name,
#     config=custom_chunk_config
# )

print("위와 같이 'chunking_config'를 설정하여 업로드할 수 있습니다.")
print("\n(참고) 지원 파일: PDF, TXT, DOCX, HWP(hwp, hwp-v5), JSON, Python(.py), HTML 등 매우 다양합니다!")
print("(참고) 무료 등급(Free Tier)의 총 저장소 용량은 1GB입니다.")

## 9단계: "기억 저장소" 삭제하기 (Clean-up)

다 쓴 '기억 저장소'는 삭제해서 깔끔하게 정리합시다! (무료 용량 1GB는 소중하니까요)

In [ ]:
# 1 문서 목록
docs = list(
    client.file_search_stores.documents.list(
        parent=file_search_store.name
    )
)

print("문서 목록")
for d in docs:
    print(d.name)

# 2 문서 강제 삭제
for d in docs:
    print("삭제", d.name)
    client.file_search_stores.documents.delete(
        name=d.name,
        config={"force": True}   # 청크까지 같이 삭제
    )

# 3 스토어도 강제 삭제
print("스토어 삭제", file_search_store.name)
client.file_search_stores.delete(
    name=file_search_store.name,
    config={"force": True}
)

# 4 확인
print("남은 스토어")
for fs in client.file_search_stores.list():
    print(fs.name)


---
## 🥳 축하합니다! '제미'에게 파일 읽기 마스터!

이제 여러분은 '제미'에게 PDF, HWP, TXT 등 원하는 파일을 읽히고, 그 내용에 대해서만 대답하는 '전문가 챗봇'을 만들 수 있게 되었습니다!

아래 10개의 연습 문제를 풀면서 '파일 검색' 기능을 마스터해 보세요!

### 1. '수업 자료' 저장소 만들기

`display_name`이 '수업 자료'인 '기억 저장소'를 새로 만들어보세요.

In [ ]:
client = genai.Client()
my_store = client.file_search_stores.____(
    config={'display_name': '____'}
)
print(f"'{my_store.display_name}' 저장소 생성 완료! (ID: {my_store.name})")

### 2. '파이썬'에 대한 메모 파일 만들고 업로드하기

먼저 'python.txt' 파일을 만들고, `upload_to_file_search_store`로 '수업 자료' 저장소에 업로드해 보세요.

In [ ]:
# 1. python.txt 파일 생성
with open("python.txt", "w") as f:
    f.write("파이썬(Python)은 1991년에 귀도 반 로섬이 발표한 프로그래밍 언어입니다.")
print("'python.txt' 파일 생성 완료")

# 2. '수업 자료' 저장소에 업로드 (my_store 변수 사용)
operation = client.file_search_stores.____(
    file='____', # 업로드할 파일 이름
    file_search_store_name=____.____, # '수업 자료' 저장소의 고유 ID 이름
    config={'display_name': '파이썬 소개'}
)

# 3. (중요!) 처리가 끝날 때까지 기다리기
while not operation.____:
    print("...파이썬 파일 읽는 중...")
    time.sleep(2)
    operation = client.operations.get(operation)

print("🎉 파이썬 파일 처리 완료!")

### 3. '파이썬'에 대해 질문하기 (RAG)

`tools`를 설정해서 '제미'가 '수업 자료' 저장소를 검색해 대답하도록 해보세요.

In [ ]:
model = genai.GenerativeModel('gemini-2.5-flash')

# 'my_store' 저장소를 검색하도록 'tools' 설정
tools = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[my_store.____] # 빈칸 채우기
        )
    )
]

response = model.generate_content(
    "파이썬은 언제 발표되었나요?",
    config=types.GenerateContentConfig(tools=____) # 빈칸 채우기
)

print(response.text)

### 4. 답변의 '근거(인용)' 확인하기

방금 받은 답변 `response`에서 `grounding_metadata`를 출력해 보세요.

In [ ]:
print("--- 답변 근거 --- ")
print(response.candidates[0].____)

### 5. '자바' 파일 만들고 '태그' 붙여서 '가져오기'

'java.txt' 파일을 만들고, `client.files.upload`로 업로드한 뒤, `import_file`로 '수업 자료' 저장소에 '태그'를 붙여 가져와 보세요.

In [ ]:
# 1. java.txt 파일 생성
with open("java.txt", "w") as f:
    f.write("자바(Java)는 1995년에 썬 마이크로시스템즈가 발표했습니다.")
print("'java.txt' 파일 생성 완료")

# 2. '임시' 파일로 업로드
java_file_temp = client.files.____(file='java.txt', config={'name': 'Java Intro'})
print(f"임시 파일 업로드 완료: {java_file_temp.name}")

# 3. '태그'와 함께 '가져오기'
op = client.file_search_stores.____(
    file_search_store_name=my_store.name,
    file_name=java_file_temp.name,
    custom_metadata=[
        {"key": "language", "string_value": "Java"}, # 태그 1
        {"key": "year", "numeric_value": 1995}      # 태그 2
    ]
)

while not op.done: time.sleep(2); op = client.operations.get(op)
print("🎉 자바 파일 태그 붙여서 처리 완료!")

### 6. '태그 필터'로 '자바' 파일만 검색하기

`metadata_filter`를 사용해서 'language' 태그가 'Java'인 파일만 검색해 보세요.

In [ ]:
tools_java_only = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[my_store.name],
            metadata_filter="____='____'" # 빈칸을 채워 'language'가 'Java'인 파일만 검색
        )
    )
]

# 1. 파이썬에 대해 물어보기 (검색되면 안 됨)
response1 = model.generate_content("파이썬은 언제 나왔어?", config=types.GenerateContentConfig(tools=tools_java_only))
print("--- (필터: Java) 파이썬 질문 ---")
print(f"🤖: {response1.text}")

# 2. 자바에 대해 물어보기 (검색되어야 함)
response2 = model.generate_content("자바는 언제 나왔어?", config=types.GenerateContentConfig(tools=tools_java_only))
print("\n--- (필터: Java) 자바 질문 ---")
print(f"🤖: {response2.text}")

### 7. (도전) '1995년'에 나온 언어 검색하기

`metadata_filter`는 숫자(numeric) 비교도 가능해요. `"year=1995"` 필터를 만들어보세요.

In [ ]:
tools_1995_only = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[my_store.name],
            metadata_filter="____=____" # 빈칸을 채워 'year'가 1995인 파일만 검색
        )
    )
]

response = model.generate_content("1995년에 무슨 언어가 나왔어?", config=types.GenerateContentConfig(tools=tools_1995_only))
print(f"🤖: {response.text}")

### 8. 파일 '조각(Chunk)' 아주 잘게 나누기

`chunking_config`를 설정해서, 파일을 50 토큰 크기로 자르고, 5 토큰씩 겹치게 만들어보세요.

In [ ]:
small_chunk_config = {
    'chunking_config': {
        'white_space_config': {
            'max_tokens_per_chunk': ____, # 50 토큰
            'max_overlap_tokens': ____  # 5 토큰
        }
    }
}
print("설정 완료! 이 config를 upload... 함수의 'config' 매개변수로 넘겨주면 됩니다.")
print(small_chunk_config)

### 9. '기억 저장소' 목록 확인하기

지금까지 만든 '기억 저장소' 목록을 `client.file_search_stores.list()`로 확인해 보세요. (방금 만든 '수업 자료' 저장소가 보여야 해요!)

In [ ]:
print("--- 내 '기억 저장소' 목록 ---")
for fs in client.file_search_stores.____():
    print(f"- {fs.display_name} (ID: {fs.name})")

### 10. '수업 자료' 저장소 삭제하기

연습이 끝났으니 '수업 자료' 저장소(`my_store`)를 삭제해서 용량을 확보하세요.

In [ ]:
print(f"'{my_store.display_name}' 저장소를 삭제합니다...")
client.file_search_stores.____(name=my_store.____)
print("삭제 완료!")